# Fine-tuning Laya for dealership call routing

Adapts the official Laya RLCD fine-tuning notebook (`NandhaKishorM/laya`, Apache-2.0) to this
project's dealership routing task. The training script itself is **unchanged** — only the data
preparation differs.

**Notebook settings (right sidebar → Notebook options):**
* Accelerator: **GPU T4 x2**
* Internet: **On**

**Before running:** package the complete training inputs from a checkout:

```bash
uv run python training/make_kaggle_dataset.py --owner YOUR_KAGGLE_USERNAME
```

Upload the generated `kaggle/jev-dealership-data/` directory as a Kaggle dataset and attach it
(Add Data → Your datasets). Do not upload only `training/` or `synthetic.jsonl`; the notebook
requires the severity, acceptance, profile, validation, and recipe files as well.


In [ ]:
!nvidia-smi || echo "nvidia-smi unavailable"
import os, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()} | visible GPUs: {n_gpu}")
assert n_gpu >= 1, (
    "No GPU is attached to this run.\n"
    "  - Set Accelerator to 'GPU T4 x2' (or a single T4) in the right sidebar, and\n"
    "  - make sure your Kaggle account is phone-verified, which is required for accelerators.\n"
    "If you pushed this kernel with the CLI, also pass --accelerator."
)
if n_gpu < 2:
    print("!! only 1 GPU - training will run single-process, which is slower but works")

# DDP across whatever we actually got, so a single-T4 run still completes.
NPROC = max(1, n_gpu)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"ready: {n_gpu} GPU(s), torchrun nproc_per_node={NPROC}")


In [ ]:
# Primary versions match the recorded v7 Kaggle run. Kaggle supplies the CUDA-enabled torch build.
!pip install -q "laya==0.3.9" "transformers==5.17.0" "pandas==3.0.6" "scipy==1.18.1" safetensors huggingface_hub pyarrow accelerate
import laya, transformers, torch
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)


## 1. Locate the uploaded data

The dataset needs `store_profile.json`, `build_items.py`, `synthetic.jsonl`,
`severity_train.jsonl` and `acceptance_train.jsonl`. We search the common
Kaggle mount points rather than hard-coding a slug.


In [ ]:
import os, glob, shutil

WANTED = ("synthetic.jsonl", "severity_train.jsonl", "acceptance_train.jsonl", "store_profile.json", "build_items.py", "train_ddp.py", "validation.py", "run_config.json")

def log_tree(root, limit=40):
    print(f"  tree under {root}:")
    n = 0
    for dirpath, dirnames, filenames in os.walk(root):
        for f in filenames:
            p = os.path.join(dirpath, f)
            print(f"    {os.path.relpath(p, root)}  {os.path.getsize(p):,} bytes")
            n += 1
            if n >= limit:
                print("    ...")
                return
    if n == 0:
        print("    (empty)")

print("=== /kaggle/input ===")
if os.path.isdir("/kaggle/input"):
    entries = sorted(os.listdir("/kaggle/input"))
    print("  entries:", entries or "(none)")
    for e in entries:
        log_tree(os.path.join("/kaggle/input", e), limit=10)
else:
    print("  /kaggle/input does not exist")

# Find the files anywhere under /kaggle/input rather than assuming a mount layout.
found_dir = None
for dirpath, _, filenames in os.walk("/kaggle/input"):
    if all(w in filenames for w in WANTED):
        found_dir = dirpath
        break

if found_dir is None:
    raise SystemExit(
        "Could not find all of {0} under /kaggle/input.\n"
        "Attach the dataset generated by training/make_kaggle_dataset.py "
        "(Add Data → Your datasets) and re-run.".format(WANTED)
    )

print(f"\nusing data from {found_dir}")
for name in WANTED:
    shutil.copy(os.path.join(found_dir, name), os.path.join("/kaggle/working", name))
    print("  copied", name)

n = sum(1 for _ in open("/kaggle/working/synthetic.jsonl"))
print(f"\n{n} labelled training utterances")


## 2. Build training items

Turns each labelled utterance into sequences for the two questions the cascade asks at inference:
`destination` (7 options in the shipped profile) and `subqueue` (the branch for the gold
destination). Unmodified `train_ddp.py` then consumes `train_items.pt`.

If the option-budget warning below fires, some rows were skipped — that is the `head_max_len`
constraint, and it is reported rather than hidden.


In [ ]:
!cd /kaggle/working && JEV_STRICT_BUILD=1 python build_items.py /kaggle/working/synthetic.jsonl /kaggle/working/train_items.pt


## 3. Fine-tune (RLCD, DDP across both T4s)

The maintainer's training script, vendored verbatim at `training/train_ddp.py`. This recipe runs
8 epochs; expect roughly 5–20 minutes depending on dataset size.


In [ ]:
MODEL_DIR = "/kaggle/working/laya_base"
MODEL_ID = "convaiinnovations/laya"
import os
MODEL_REVISION = os.environ.get("LAYA_MODEL_REVISION", "main")
from huggingface_hub import snapshot_download
snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir=MODEL_DIR)
os.environ.setdefault("LAYA_MODEL_ID", MODEL_ID)
os.environ.setdefault("LAYA_MODEL_REVISION", MODEL_REVISION)
print("base checkpoint at", MODEL_DIR, "revision", MODEL_REVISION)

# The recipe, from training/run_config.json. train_ddp.py reads the environment, so this cell is
# what carries it across; there is no second copy to forget.
EPOCHS = 8
LR_ENCODER = 2.5e-05
LR_HEAD = 0.0001
os.environ["JEV_EPOCHS"] = str(EPOCHS)
os.environ["JEV_LR_ENCODER"] = str(LR_ENCODER)
os.environ["JEV_LR_HEAD"] = str(LR_HEAD)
os.environ["JEV_SEED"] = str(42)
os.environ["JEV_MAX_LEN"] = str(512)
os.environ["JEV_HEAD_MAX_LEN"] = str(192)
os.environ.setdefault("JEV_REQUIRE_SPLITS", "0")
print(f"training for {EPOCHS} epochs at lr {LR_ENCODER}/{LR_HEAD} with seed {os.environ['JEV_SEED']}")

OUTPUT_DIR = "/kaggle/working/laya-dealership-routing"
cmd = (f"torchrun --standalone --nproc_per_node={NPROC} /kaggle/working/train_ddp.py "
       f"{MODEL_DIR} {OUTPUT_DIR} /kaggle/working/train_items.pt")
print("running:", cmd)
!{cmd}


## 4. Package the result

Zips the fine-tuned checkpoint so it can be downloaded and evaluated **locally** with the project's
own harness — `scripts/eval.py` — rather than re-implementing the evaluation here. One eval
harness, one source of truth.


In [ ]:
import os, shutil
OUTPUT_DIR = "/kaggle/working/laya-dealership-routing"
assert os.path.isdir(OUTPUT_DIR), f"{OUTPUT_DIR} does not exist - training did not finish"
shutil.make_archive("/kaggle/working/laya-dealership-routing", "zip", OUTPUT_DIR)
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.relpath(p, OUTPUT_DIR):50s} {os.path.getsize(p):>12,} bytes")
print("\nzip ready: /kaggle/working/laya-dealership-routing.zip")


## Next: evaluate it properly

1. Download `laya-dealership-routing.zip` from the notebook output.
2. Unzip it locally into `models/laya-dealership-routing/`.
3. Run the existing evaluation against it:

```bash
uv run python scripts/eval.py --limit 0 --determinism 1
```

The held-out 81 hand-labelled cases were **never** in this training set — `build_items.py` only
reads `synthetic.jsonl`, and the generator's deduper refuses anything within Jaccard 0.6 of them.
